# ML-07 ? Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane: **Content Refresh / refresh prioritization**. This notebook builds a simple, transparent baseline queue that Week 5 modeling should beat. All numbers below come from executed code on the bundled anonymized content-refresh dataset.


## 1. Load data and choose safe baseline inputs

The baseline uses only decision-moment inputs: staleness/update age, current 90-day search visibility, keyword demand, CTR, and current average position. It does **not** use `trend_direction`, `trend_pct`, `is_declining_label`, future-window columns, or any refresh/outcome label.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 140)

DATA_PATHS = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
]
DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv in the expected repo paths.")

df = pd.read_csv(DATA_PATH)

numeric_cols = [
    "search_volume", "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "days_since_last_update", "content_age_days",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print(f"Columns: {len(df.columns)}")
display(df[["content_id", "search_volume", "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update", "freshness_tier", "position_tier"]].head())


Loaded 30,000 rows from data\raw\content_refresh_anonymized.csv
Columns: 44


,content_id,search_volume,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,freshness_tier,position_tier
0,content_304f48230142,10.0,3803,29,0.76,10.6,20,0-30,striking
1,content_a1fb4e703a9e,90.0,15320,7,0.05,20.3,25,0-30,page_3_5
2,content_9aa793d4d895,0.0,12581,11,0.09,36.5,20,0-30,page_3_5
3,content_331d6c4de07b,10.0,11751,58,0.49,6.2,22,0-30,page_1
4,content_d99b7a2d90ca,0.0,19140,24,0.13,44.0,14,0-30,page_3_5


## 2. Two signal checks before encoding the rule

The two checks below use signals the baseline rule actually relies on. Signal 1 is explicitly tied to the FlyRank refresh/staleness flag logic: pages that have not been updated recently may be better refresh candidates. Signal 2 is tied to CTR-fix logic: pages with visibility but weak CTR are candidates for title/meta/content review.

### Signal 1 ? Staleness behind refresh flags

Bucket definition: `freshness_tier` from `days_since_last_update` (`0-30`, `31-90`, `91-180`, `181+`, `never`). The table shows `n`, the median days since update, and opportunity/performance aggregates.


In [2]:
freshness_order = ["0-30", "31-90", "91-180", "181+", "never"]
staleness_check = df.copy()
staleness_check["staleness_bucket"] = pd.Categorical(
    staleness_check["freshness_tier"], categories=freshness_order, ordered=True
)

staleness_table = (
    staleness_check
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_days_since_last_update=("days_since_last_update", "median"),
        median_search_volume=("search_volume", "median"),
        median_impressions_90d=("impressions_90d", "median"),
        median_ctr_pct=("ctr", "median"),
    )
    .reset_index()
)

display(staleness_table)

ctr_0_30 = staleness_table.loc[staleness_table["staleness_bucket"].astype(str).eq("0-30"), "median_ctr_pct"].iloc[0]
ctr_91_180 = staleness_table.loc[staleness_table["staleness_bucket"].astype(str).eq("91-180"), "median_ctr_pct"].iloc[0]
impr_91_180 = staleness_table.loc[staleness_table["staleness_bucket"].astype(str).eq("91-180"), "median_impressions_90d"].iloc[0]
impr_181 = staleness_table.loc[staleness_table["staleness_bucket"].astype(str).eq("181+"), "median_impressions_90d"].iloc[0]
staleness_verdict = "MIXED" if (ctr_91_180 > ctr_0_30 and impr_181 < impr_91_180) else "CONFIRMED"
print(f"Verdict: {staleness_verdict}")
print(
    "Why: older 91-180 day pages have stronger median impressions and CTR than the freshest bucket, "
    "but the 181+ bucket has very low median impressions and CTR. Staleness helps identify refresh candidates, "
    "yet age alone is not enough."
)


Verdict: MIXED
Why: older 91-180 day pages have stronger median impressions and CTR than the freshest bucket, but the 181+ bucket has very low median impressions and CTR. Staleness helps identify refresh candidates, yet age alone is not enough.


,staleness_bucket,n,median_days_since_last_update,median_search_volume,median_impressions_90d,median_ctr_pct
0,0-30,20480,20.0,10.0,470.0,0.04
1,31-90,175,41.0,10.0,510.0,0.00
2,91-180,9171,104.0,10.0,1692.0,0.10
3,181+,174,211.0,0.0,15.5,0.00
4,never,0,NaN,NaN,NaN,NaN


### Signal 2 ? CTR versus ranking position

Bucket definition: `position_tier` after applying a volume floor of `impressions_90d >= 100` so that tiny-denominator CTRs do not dominate. The relevant statistic is median/mean CTR by ranking bucket.


In [3]:
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
position_check = df[df["impressions_90d"] >= 100].copy()
position_check["position_bucket"] = pd.Categorical(
    position_check["position_tier"], categories=position_order, ordered=True
)

position_ctr_table = (
    position_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_avg_position=("avg_position", "median"),
        median_impressions_90d=("impressions_90d", "median"),
        median_ctr_pct=("ctr", "median"),
        mean_ctr_pct=("ctr", "mean"),
    )
    .reset_index()
)

display(position_ctr_table)

page_1_ctr = position_ctr_table.loc[position_ctr_table["position_bucket"].astype(str).eq("page_1"), "median_ctr_pct"].iloc[0]
striking_ctr = position_ctr_table.loc[position_ctr_table["position_bucket"].astype(str).eq("striking"), "median_ctr_pct"].iloc[0]
page_3_5_ctr = position_ctr_table.loc[position_ctr_table["position_bucket"].astype(str).eq("page_3_5"), "median_ctr_pct"].iloc[0]
deep_ctr = position_ctr_table.loc[position_ctr_table["position_bucket"].astype(str).eq("deep"), "median_ctr_pct"].iloc[0]
ctr_position_verdict = "CONFIRMED" if page_1_ctr > striking_ctr > page_3_5_ctr >= deep_ctr else "MIXED"
print(f"Verdict: {ctr_position_verdict}")
print(
    "Why: with at least 100 impressions, median CTR falls from page-1 to striking to page-3/5 to deep positions. "
    "The top-3 median is slightly below page-1 in this slice, so the rule uses CTR as an opportunity flag, "
    "not as a perfect ranking model."
)


Verdict: CONFIRMED
Why: with at least 100 impressions, median CTR falls from page-1 to striking to page-3/5 to deep positions. The top-3 median is slightly below page-1 in this slice, so the rule uses CTR as an opportunity flag, not as a perfect ranking model.


,position_bucket,n,median_avg_position,median_impressions_90d,median_ctr_pct,mean_ctr_pct
0,top_3,533,2.4,2918.0,0.19,0.334128
1,page_1,8633,6.6,2945.0,0.23,0.354760
2,striking,5903,14.0,1388.0,0.15,0.255782
3,page_3_5,6058,28.8,1210.5,0.06,0.142359
4,deep,879,59.5,426.0,0.00,0.055415
5,no_data,0,NaN,NaN,NaN,NaN


## 3. One transparent baseline rule

Plain-English rule: prioritize pages for refresh review when they are stale, have measurable search visibility, show weak CTR for their ranking bucket, or have meaningful keyword demand. This is a hand-written score, not a trained model.

Reason-code priority is explicit and each row receives exactly one primary reason: `HIGH_STALENESS` first, then `LOW_CTR`, then `HIGH_DEMAND`, else `LOWER_OPPORTUNITY`. The action label is `REVIEW_FOR_REFRESH` for the top scoring quarter of rows and `LOWER_PRIORITY` otherwise.


In [4]:
data = df.copy()

safe_baseline_inputs = [
    "days_since_last_update",
    "freshness_tier",
    "impressions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "position_tier",
]

demand_threshold = data["search_volume"].quantile(0.75)
visibility_threshold = 300  # data dictionary: moderate impression tier starts at 300

position_ctr_threshold = (
    data[data["impressions_90d"] >= 100]
    .groupby("position_tier")["ctr"]
    .median()
    .to_dict()
)
global_ctr_threshold = data.loc[data["impressions_90d"] >= 100, "ctr"].median()

data["is_stale"] = data["days_since_last_update"] >= 91
data["has_visibility"] = data["impressions_90d"] >= visibility_threshold
data["has_high_demand"] = data["search_volume"].fillna(0) >= demand_threshold
data["ctr_threshold_for_bucket"] = data["position_tier"].map(position_ctr_threshold).fillna(global_ctr_threshold)
data["has_low_ctr"] = (
    data["has_visibility"]
    & data["avg_position"].gt(0)
    & data["ctr"].le(data["ctr_threshold_for_bucket"])
)

log_impressions = np.log1p(data["impressions_90d"].fillna(0))
data["visibility_component"] = log_impressions / log_impressions.max()

data["baseline_score"] = (
    data["is_stale"].astype(int) * 40
    + data["has_low_ctr"].astype(int) * 30
    + data["has_high_demand"].astype(int) * 20
    + data["has_visibility"].astype(int) * 5
    + data["visibility_component"] * 5
)

def primary_reason(row):
    if row["is_stale"]:
        return "HIGH_STALENESS"
    if row["has_low_ctr"]:
        return "LOW_CTR"
    if row["has_high_demand"]:
        return "HIGH_DEMAND"
    return "LOWER_OPPORTUNITY"

data["reason_code"] = data.apply(primary_reason, axis=1)
review_threshold = data["baseline_score"].quantile(0.75)
data["action"] = np.where(data["baseline_score"] >= review_threshold, "REVIEW_FOR_REFRESH", "LOWER_PRIORITY")

ranked = data.sort_values(
    ["baseline_score", "impressions_90d", "search_volume"], ascending=[False, False, False]
).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

output_cols = [
    "rank", "content_id", "baseline_score", "reason_code", "action",
    "days_since_last_update", "freshness_tier", "impressions_90d", "search_volume",
    "ctr", "avg_position", "position_tier", "is_stale", "has_low_ctr", "has_high_demand",
]
queue = ranked[output_cols].copy()

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print(f"Demand threshold used: {demand_threshold:.2f}")
print(f"Review threshold used: {review_threshold:.2f}")
print(f"Ranked rows: {len(queue):,}")
print(f"Output written to: {output_path}")
print(f"Maximum number of reason codes per row: {queue["reason_code"].str.count("\\|").add(1).max()}")
display(queue.head(20))


Demand threshold used: 20.00
Review threshold used: 49.14
Ranked rows: 30,000
Output written to: work\outputs\baseline_action_score.csv
Maximum number of reason codes per row: 1


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,freshness_tier,impressions_90d,search_volume,ctr,avg_position,position_tier,is_stale,has_low_ctr,has_high_demand
0,1,content_5fe46e04994d,100.000000,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,517715,1900.0,0.14,4.2,page_1,True,True,True
1,2,content_cb112fce36be,99.804996,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,309910,70.0,0.16,5.6,page_1,True,True,True
2,3,content_c8e9d6ab9013,99.654702,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,208678,20.0,0.00,9.7,page_1,True,True,True
3,4,content_bb5bd5f771dc,99.590620,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,176296,20.0,0.23,4.3,page_1,True,True,True
4,5,content_f42eb861c6dd,99.535435,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,152467,20.0,0.13,6.5,page_1,True,True,True
5,6,content_11fcfd65d94c,99.526906,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,149083,480.0,0.15,6.2,page_1,True,True,True
6,7,content_97a86caf3a3d,99.523287,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,147670,40.0,0.07,6.4,page_1,True,True,True
7,8,content_cd1b913fa942,99.505307,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,140846,70.0,0.23,5.3,page_1,True,True,True
8,9,content_c1fe78bc4e37,99.486528,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,134055,70.0,0.03,7.5,page_1,True,True,True
9,10,content_45fb95832c96,99.464883,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,126633,20.0,0.12,7.6,page_1,True,True,True


## 4. Top-10 review

Each of the top 10 rows is reviewed as a human queue item: action, why the rule selected it, and what could make the recommendation wrong.


In [5]:
why_templates = {
    "HIGH_STALENESS": "Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.",
    "LOW_CTR": "Visible page with CTR at or below its position-bucket median; selected by CTR-fix logic.",
    "HIGH_DEMAND": "Search volume is at or above the dataset 75th percentile; selected by demand logic.",
    "LOWER_OPPORTUNITY": "No primary opportunity flag fired; included only if visibility score is high enough.",
}
wrong_templates = {
    "HIGH_STALENESS": "The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged.",
    "LOW_CTR": "CTR can be depressed by SERP layout, brand intent, snippets, or measurement noise rather than a fixable content issue.",
    "HIGH_DEMAND": "High search volume may not match this page intent, or the keyword may be too broad to justify a refresh.",
    "LOWER_OPPORTUNITY": "A simple rule may miss business context, seasonality, or a recent manual change not represented in the data.",
}

top10_review = queue.head(10).copy()
top10_review["why_it_is_here"] = top10_review["reason_code"].map(why_templates)
top10_review["what_would_make_it_wrong"] = top10_review["reason_code"].map(wrong_templates)

display(top10_review[["rank", "content_id", "action", "reason_code", "why_it_is_here", "what_would_make_it_wrong"]])

print("Top-10 action counts:")
print(top10_review["action"].value_counts())
print("\nTop-10 reason codes:")
print(top10_review["reason_code"].value_counts())


Top-10 action counts:
action
REVIEW_FOR_REFRESH    10
Name: count, dtype: int64

Top-10 reason codes:
reason_code
HIGH_STALENESS    10
Name: count, dtype: int64


,rank,content_id,action,reason_code,why_it_is_here,what_would_make_it_wrong
0,1,content_5fe46e04994d,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
1,2,content_cb112fce36be,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
2,3,content_c8e9d6ab9013,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
3,4,content_bb5bd5f771dc,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
4,5,content_f42eb861c6dd,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
5,6,content_11fcfd65d94c,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
6,7,content_97a86caf3a3d,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
7,8,content_cd1b913fa942,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
8,9,content_c1fe78bc4e37,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."
9,10,content_45fb95832c96,REVIEW_FOR_REFRESH,HIGH_STALENESS,Stale page: days_since_last_update is 91+; score is boosted by refresh-age logic.,"The page may have been updated outside the tracked field, or the content may be evergreen and intentionally unchanged."


## 5. Top-20 queue preview

The assignment allows keeping a Top-20 output. This preview is useful for scanning the ranked queue after the required Top-10 review.


In [6]:
display(queue.head(20))
print("Top-20 action counts:")
print(queue.head(20)["action"].value_counts())
print("\nTop-20 reason codes:")
print(queue.head(20)["reason_code"].value_counts())


Top-20 action counts:
action
REVIEW_FOR_REFRESH    20
Name: count, dtype: int64

Top-20 reason codes:
reason_code
HIGH_STALENESS    20
Name: count, dtype: int64


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,freshness_tier,impressions_90d,search_volume,ctr,avg_position,position_tier,is_stale,has_low_ctr,has_high_demand
0,1,content_5fe46e04994d,100.000000,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,517715,1900.0,0.14,4.2,page_1,True,True,True
1,2,content_cb112fce36be,99.804996,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,309910,70.0,0.16,5.6,page_1,True,True,True
2,3,content_c8e9d6ab9013,99.654702,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,208678,20.0,0.00,9.7,page_1,True,True,True
3,4,content_bb5bd5f771dc,99.590620,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,176296,20.0,0.23,4.3,page_1,True,True,True
4,5,content_f42eb861c6dd,99.535435,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,152467,20.0,0.13,6.5,page_1,True,True,True
5,6,content_11fcfd65d94c,99.526906,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,149083,480.0,0.15,6.2,page_1,True,True,True
6,7,content_97a86caf3a3d,99.523287,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,147670,40.0,0.07,6.4,page_1,True,True,True
7,8,content_cd1b913fa942,99.505307,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,140846,70.0,0.23,5.3,page_1,True,True,True
8,9,content_c1fe78bc4e37,99.486528,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,134055,70.0,0.03,7.5,page_1,True,True,True
9,10,content_45fb95832c96,99.464883,HIGH_STALENESS,REVIEW_FOR_REFRESH,104,91-180,126633,20.0,0.12,7.6,page_1,True,True,True


## 6. Weak picks and leakage check

Weak picks are examples where the simple baseline may produce questionable recommendations. The point is not to remove them by hand, but to understand the limits of a transparent rule before Week 5 modeling.

Leakage statement: the score is built only from staleness, current visibility, keyword demand, CTR, and current position. It excludes future-window fields and label-derived fields such as `trend_direction`, `trend_pct`, `is_declining_label`, and any refresh-priority target.


In [7]:
review_rows = queue[queue["action"].eq("REVIEW_FOR_REFRESH")].copy()
weak_picks = (
    review_rows
    .assign(
        possible_issue=np.select(
            [
                ~review_rows["has_low_ctr"] & review_rows["reason_code"].eq("HIGH_STALENESS"),
                review_rows["impressions_90d"].lt(visibility_threshold * 2),
                review_rows["search_volume"].fillna(0).eq(0),
            ],
            [
                "Selected mainly because it is stale; CTR is not weak for its position bucket.",
                "Selected despite relatively modest visibility, so the business impact may be small.",
                "Selected with missing/zero keyword demand, so the target keyword context may be weak.",
            ],
            default="Simple rule may miss page intent, SERP features, seasonality, or recent edits.",
        )
    )
    .sort_values(["baseline_score", "impressions_90d"], ascending=[True, True])
    .head(10)
)

display(weak_picks[["rank", "content_id", "baseline_score", "action", "reason_code", "possible_issue"]])

blocked_terms = ["future", "next", "label", "target", "declining", "trend", "refresh_priority"]
leakage_inputs = [col for col in safe_baseline_inputs if any(term in col.lower() for term in blocked_terms)]
label_derived_inputs_used = [col for col in safe_baseline_inputs if col in {"trend_direction", "trend_pct", "is_declining_label", "refresh_priority"}]

print("Baseline inputs:")
print(safe_baseline_inputs)
print("\nFuture-window inputs used:")
print(leakage_inputs)
print("\nLabel-derived inputs used:")
print(label_derived_inputs_used)

assert not leakage_inputs, "Future-window or trend-like input detected."
assert not label_derived_inputs_used, "Label-derived input detected."
assert queue["reason_code"].str.contains("\\|").sum() == 0, "A concatenated reason code was found."
assert output_path.exists(), "baseline_action_score.csv was not written."

print("\nLeakage check passed: no future-window or label-derived inputs are used, and every row has one primary reason code.")


Baseline inputs:
['days_since_last_update', 'freshness_tier', 'impressions_90d', 'search_volume', 'ctr', 'avg_position', 'position_tier']

Future-window inputs used:
[]

Label-derived inputs used:
[]

Leakage check passed: no future-window or label-derived inputs are used, and every row has one primary reason code.


,rank,content_id,baseline_score,action,reason_code,possible_issue
7499,7500,content_12008fddfdbf,49.148476,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7498,7499,content_90bb37b53856,49.156931,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7497,7498,content_6109e35fbc01,49.159922,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7496,7497,content_61873e150f71,49.160851,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7495,7496,content_a05d66a41827,49.162058,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7494,7495,content_884004792611,49.169351,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7493,7494,content_2464e067f604,49.170069,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7492,7493,content_22d8297ff88a,49.177275,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7491,7492,content_7368877ea310,49.177672,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.
7490,7491,content_c2745ea9f931,49.193179,REVIEW_FOR_REFRESH,HIGH_STALENESS,Selected mainly because it is stale; CTR is not weak for its position bucket.


## Self-check

- [x] Two signals were checked before encoding the rule.
- [x] Each signal has a visible bucket table.
- [x] Each bucket table prints `n`.
- [x] At least one signal is linked to a real FlyRank flag.
- [x] Each signal has exactly one verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.
- [x] One transparent baseline rule is encoded.
- [x] Each row has one primary reason code.
- [x] Each row has an action label.
- [x] Ranked queue is generated.
- [x] `work/outputs/baseline_action_score.csv` is written.
- [x] Top 10 rows are individually reviewed.
- [x] Each top-10 row has action, why, and what would make it wrong.
- [x] No future-window inputs are used.
- [x] No label-derived inputs are used.
- [x] Notebook executes from top to bottom.
